In [0]:
TABLE_PRODUCT_SILVER = "customer_360.silver.products"
TABLE_PRODUCT_DIM = "customer_360.dim.dim_product"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_PRODUCT_DIM} (
    product_sk BIGINT NOT NULL,
    product_id STRING NOT NULL,
    product_name STRING,
    product_category STRING,
    product_subcategory STRING,
    product_price DECIMAL(12,2),
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA
""")

In [0]:
product_silver_df=spark.read.format("delta").table(TABLE_PRODUCT_SILVER)
product_dim=spark.read.format("delta").table(TABLE_PRODUCT_DIM)
# display(product_silver_df)
# display(product_dim)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Find product versions that do not already exist in the dimension
new_products = (
    product_silver_df.alias("ps")
    .join(
        product_dim.alias("pd"),
        (col("ps.product_id") == col("pd.product_id")) &
        (col("ps.updated_at") == col("pd.effective_from")),
        how="left_anti"
    )
)

# Get the current maximum surrogate key
max_sk = (
    product_dim
    .agg(max("product_sk").alias("max_sk"))
    .collect()[0]["max_sk"]
)

if max_sk is None:
    max_sk = 0

# Generate surrogate keys for new records
w = Window.orderBy("product_id", "updated_at")

new_products = (
    new_products
    .withColumn(
        "product_sk",
        row_number().over(w) + lit(max_sk)
    )
    .withColumn(
        "effective_from",
        current_timestamp()
    )
    .withColumn(
        "effective_to",
        to_timestamp(lit("9999-12-31 23:59:59"))
    )
    .withColumn(
        "is_current",
        lit(True)
    )
    .select(
        "product_sk",
        "product_id",
        "product_name",
        "product_category",
        "product_subcategory",
        "product_price",
        "effective_from",
        "effective_to",
        "is_current"
    )
)

# display(new_products)

In [0]:
(
    new_products
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_PRODUCT_DIM)
    )


In [0]:
display(spark.sql("""
SELECT *
FROM customer_360.dim.dim_product
"""))